# Value leakage on Qwen3.5 — where does the value intervene?

**What this notebook does, in one paragraph.** We ask a small open model a question with a numeric answer nobody can check, and add a note saying that if the number lands on one side of a threshold, money goes to a charity the model probably likes better. If the model's number drifts toward that side, its own values leaked into an answer that was supposed to be about the world. We first measure that drift (the *go/no-go*). Then we look inside the model for a direction in its activations that carries "which side is the good side", and we remove that direction while the model writes its answer. If the drift disappears, that direction is where the value gets in. If it doesn't, we learn the value lives somewhere else. Both are results.

**How to use it.** Run the cells top to bottom. Every code cell is preceded by a note saying what it does, why, and what to look for. Settings live in one cell (2) so you never edit the commands. The clock (Nanda's 16–20 h) starts at the cell marked ⏱; everything above it is setup.

**Where things go.** Every model answer is saved word for word to `data/raw/` (one file per run). Summary numbers go to `data/processed/`. Your notes go in `journal/`. All three live on your Google Drive under `mats12_runs`, so nothing is lost when Colab's machine disappears.

**Vocabulary you will meet.** *Baseline* = the question with no bet. *above_good / below_good* = the two versions of the bet note (which side of the threshold is the good side). *Balanced bias* = the paper's leak score: 0 means no leak, 1 means the model always lands on the good side. *Variant* = which wording the note uses: `abstract` says "good cause / bad cause" outright; `concrete_amf_kw` names two real charities and leaves the judgement to the model; `equal_dwb_imc` names two equally good charities, so there is no reason to lean. *Direction* = a single vector in the model's internal space; *ablation* = removing that vector from the model's internal state while it generates.

### 1 · Is there a GPU?
Colab gives you a GPU only if you asked for one: **Runtime → Change runtime type → GPU**. This cell prints which card you got. A free T4 is fine (it will use 16-bit floats). If it says no GPU, fix the runtime type and run again.

In [ ]:
import torch, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU — Runtime → Change runtime type → GPU, then run this cell again")

### 2 · Settings — the only cell you edit
- `MODEL`: which open model to study. Start with the 4B; switch to 9B only if the 4B shows no leak (9B needs an L4/A100).
- `RUN`: a name for this session's files. Change it if you want a fresh set of results without overwriting the old ones.
- `N`: how many answers to sample per condition. 20 is enough to see a leak; 50 makes the confidence interval tighter.
- `LAYERS`: which layers to intervene on later (the 4B has 32; the middle-to-late band is the usual place to look). You can come back and change this after seeing the direction table.
- `DRIVE_DIR`: where results persist on your Google Drive.

In [ ]:
MODEL     = "Qwen/Qwen3.5-4B"
RUN       = "gonogo_4b"
N         = 20
LAYERS    = "16,20,24"
DRIVE_DIR = "/content/drive/MyDrive/mats12_runs"


### 3 · Get the code and connect your Drive (uncounted)
Mounts Google Drive, clones the private repo (needs `GITHUB_TOKEN` in Colab's Secrets panel, the key icon on the left), installs the Python packages, and moves `data/`, `figures/` and `journal/` onto Drive. Takes a minute or two. Colab will pop up a permission dialog for Drive; accept it.

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
from getpass import getpass
token = None
for name in ("GITHUB_TOKEN", "colab-mats12"):     # Colab Secrets panel (key icon): either name, "Notebook access" ON
    try:
        token = userdata.get(name)
        if token: break
    except Exception:
        pass
if not token:
    print("No GITHUB_TOKEN found in Colab Secrets. The repo is private, so paste a GitHub fine-grained token")
    print("(GitHub → Settings → Developer settings → Fine-grained tokens; repository: mats12; Contents: read and write).")
    token = getpass("GitHub token: ").strip()
# The token travels in a per-command header, never in the saved remote URL (so it is not written to .git/config).
import base64
AUTH = ["-c", "http.extraheader=AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
os.environ["MATS12_GIT_AUTH"] = AUTH[1]
if not os.path.exists("/content/mats12"):
    r = subprocess.run(["git", *AUTH, "clone", "-q", "https://github.com/martinherje/mats12.git", "/content/mats12"], capture_output=True, text=True)
else:
    # The Drive links make git see local changes; autostash carries them across the pull. If that still fails,
    # back up every changed file to Drive, reset, and pull clean — nothing is lost, and the backup path is printed.
    R = "/content/mats12"
    r = subprocess.run(["git", *AUTH, "-C", R, "pull", "--rebase", "--autostash", "-q"], capture_output=True, text=True)
    if r.returncode != 0:
        import shutil, time
        bk = f"{DRIVE_DIR}/backup_{time.strftime('%Y%m%d_%H%M%S')}"
        changed = subprocess.run(["git", "-C", R, "status", "--porcelain"], capture_output=True, text=True).stdout.split("\n")
        for line in changed:
            f = line[3:].strip()
            if f and os.path.isfile(os.path.join(R, f)):
                os.makedirs(os.path.dirname(os.path.join(bk, f)), exist_ok=True); shutil.copy2(os.path.join(R, f), os.path.join(bk, f))
        subprocess.run(["git", "-C", R, "rebase", "--abort"], capture_output=True)
        subprocess.run(["git", "-C", R, "reset", "--hard", "-q"], check=True)
        r = subprocess.run(["git", *AUTH, "-C", R, "pull", "-q"], capture_output=True, text=True)
        print(f"local changes were backed up to {bk} and the repo reset before pulling")
if r.returncode != 0:
    raise SystemExit("git failed: " + r.stderr.replace(token, "<token>").strip() + "\nCheck the token has access to martinherje/mats12 (Contents: read).")
%cd /content/mats12
!git log --oneline -1
!bash scripts/colab_setup.sh "$DRIVE_DIR" 

### 4 · Load the model once to check it fits (uncounted)
Downloads the model (about 8 GB for the 4B, once; it is cached on this machine) and runs a single forward pass. Look for: the layer count (32 for the 4B), the width (2560), and peak memory comfortably below what the GPU has.

In [ ]:
!python scripts/gpu_smoke.py --model $MODEL

## ⏱ The clock starts here
Before running anything below: open `journal/design-questions-value-leakage.md` in the Files panel on the left (`mats12 → journal`), answer each question in a sentence, and copy your answers into `journal/highlights.md` with today's date. That is the prediction you are testing. Start Toggl.

### 5 · Go/no-go: does the model leak? (~15–30 min on a T4)
Three runs, thinking switched off so the model answers directly:
1. **concrete**: the note names Against Malaria Foundation vs Kids Wish Network. The model has to decide which is better itself. This is the one we care about. It also samples the *baseline* (no bet) answers and sets each question's threshold at their median, so without a leak the number lands on the good side half the time.
2. **abstract**: the note says "good cause / bad cause" outright. Reuses the thresholds from run 1.
3. **equal**: two comparably good charities. Reuses the thresholds. This should show no leak; if it does, something other than values is moving the number.

Every answer is saved in full to `data/raw/donation_bet_<run>.jsonl`.

In [ ]:
!python scripts/donation_bet.py --backend local --model $MODEL --run $RUN --variant concrete_amf_kw --think off --n-baseline $N --n-per-direction $N
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_abstract --variant abstract --think off --n-per-direction $N --reuse-thresholds $RUN
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_equal --variant equal_dwb_imc --think off --n-per-direction $N --reuse-thresholds $RUN

### 6 · The headline table
One row per variant. `bias` is the leak score (0 = none), `ci_lo`/`ci_hi` its 95% interval, `unparsed` the share of answers where no single number could be read off (if that is above ~0.15, look at the raw answers before trusting anything).

**What you hope to see:** abstract ≥ concrete > equal ≈ 0, with the concrete interval clear of zero. **If concrete ≈ 0 but abstract > 0:** the model follows stated valence but brings none of its own; that is itself the first finding, and the next setting is the developer-preference task. **If everything ≈ 0:** try the 9B model (change `MODEL` in cell 2, rerun from cell 4).

In [ ]:
import json, pandas as pd
rows = []
for run, variant in [(RUN, "concrete"), (f"{RUN}_abstract", "abstract"), (f"{RUN}_equal", "equal")]:
    try:
        for think, s in json.load(open(f"data/processed/donation_bet_{run}.json"))["headline"].items():
            rows.append({"variant": variant, "think": think, "bias": s["bias"], "ci_lo": s["ci_lo"], "ci_hi": s["ci_hi"], "n": s["n"], "unparsed": s["unparsed_frac"]})
    except FileNotFoundError:
        rows.append({"variant": variant, "bias": "run missing"})
print(pd.DataFrame(rows).round(3).to_string(index=False))

### 7 · Hand-check the parsing — read these, it is the first thing Nanda asks about
Prints twenty random answers per condition, with the number the script extracted and which rule extracted it. Check the number is the model's actual final estimate. Then write one line in `journal/verification-log.md`: how many you read, how many were wrong. If more than a couple are wrong, tell Claude; the parser gets fixed before anything else runs.

In [ ]:
import json, random
random.seed(0)
rows = [json.loads(line) for line in open(f"data/raw/donation_bet_{RUN}.jsonl")]
for condition in ("baseline", "above_good", "below_good"):
    subset = [r for r in rows if r["condition"] == condition]
    print(f"\n===== {condition}  (n={len(subset)}) =====")
    for r in random.sample(subset, min(20, len(subset))):
        print(f"[{r['question']:10s}] extracted={r['estimate']!s:>14} via {r['parse_method']:15s} | ...{r['answer'][-160:].replace(chr(10), ' ')}")

### 8 · Optional: the same test with thinking ON (slow)
Same concrete note, but the model writes a chain of thought first. Fewer samples because each answer is long. **Why it matters:** if the leak is present with thinking off, the value enters inside the forward pass; if it only appears with thinking on, the model reasons its way into it. That one comparison answers "where does the value intervene" at the coarsest level. The chain of thought is saved too, so you can later read whether the model *admits* it is aiming for the good side.

In [ ]:
N_ON = 10
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_thinkon --variant concrete_amf_kw --think on --n-per-direction $N_ON --reuse-thresholds $RUN --max-new-tokens 1500
import json; print(json.load(open(f"data/processed/donation_bet_{RUN}_thinkon.json"))["headline"])

### 9 · Look inside: record the model's internal state, then find the direction
Step 1 runs each prompt (baseline, above_good, below_good, per question) through the model once and saves its internal state at the moment it is about to start answering, at every layer. Step 2 takes the average state under above_good minus the average under below_good: a single vector per layer, the *favoured-side direction*. Step 3 does the same for bet-present minus no-bet: the *topic direction*, which knows a bet exists but not which side is good. We need both to tell "removing the value" apart from "removing the awareness of the bet".

The printed table shows, per layer, how strongly each group of prompts points along the direction. The separation is trivially large where the two prompts literally differ; that is expected and not the result.

In [ ]:
!python scripts/extract_activations.py --model $MODEL --scenarios data/scenarios_{RUN}.csv --template chat --generation-prompt --enable-thinking off --run $RUN
!python scripts/make_direction.py --run $RUN --label good_side --filter "bet==1" --out direction_{RUN}_good_side
!python scripts/make_direction.py --run $RUN --label bet --out direction_{RUN}_bet

### 10 · The causal test: remove the direction while the model answers
Three reruns of the concrete condition, each with something subtracted from the model's internal state at the layers in `LAYERS`:
1. **ablate** the favoured-side direction. If the leak drops, the direction carries the value.
2. **random** direction of the same size. Should change nothing; if it also kills the leak, we were just damaging the model.
3. **topic** direction (bet vs no bet). If this kills the leak too, we removed the model's knowledge that a bet exists, which is weaker than removing which side is good.

The table at the end puts the original leak next to the three interventions. The story is in the gaps between those four numbers.

In [ ]:
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_abl   --variant concrete_amf_kw --think off --n-per-direction $N --reuse-thresholds $RUN --ablate data/processed/direction_{RUN}_good_side.npz --layers $LAYERS --mode ablate
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_rand  --variant concrete_amf_kw --think off --n-per-direction $N --reuse-thresholds $RUN --layers $LAYERS --mode random
!python scripts/donation_bet.py --backend local --model $MODEL --run {RUN}_topic --variant concrete_amf_kw --think off --n-per-direction $N --reuse-thresholds $RUN --ablate data/processed/direction_{RUN}_bet.npz --layers $LAYERS --mode ablate
import json, pandas as pd
table = [{"what": label, **json.load(open(f"data/processed/donation_bet_{run}.json"))["headline"]["off"]}
         for run, label in [(RUN, "original leak"), (f"{RUN}_abl", "favoured-side direction removed"), (f"{RUN}_rand", "random direction removed (control)"), (f"{RUN}_topic", "topic direction removed (control)")]]
print(pd.DataFrame(table)[["what", "bias", "ci_lo", "ci_hi", "unparsed_frac"]].round(3).to_string(index=False))

### 11 · Did we remove the value, or just the fact?
Asks the model directly, ten times, which of the two charities does more good, first normally, then with the favoured-side direction removed. **The result you want:** it still names the right charity after ablation (it kept the fact) while cell 10 showed the number no longer drifts (it lost the motivation). If it can no longer tell the charities apart, the ablation deleted knowledge, not motivation, and the write-up has to say so.

In [ ]:
import sys, json, numpy as np, torch; sys.path.insert(0, "scripts")
from common import load_model, pick_device, pick_dtype
from steer import Intervention
Q = json.load(open("data/donation_bet_questions.json")); v = Q["variants"]["concrete_amf_kw"]
question = Q["fact_check_prompt"].format(favoured=v["favoured"], other=v["other"])
device = pick_device("auto"); tok, model = load_model(MODEL, device, pick_dtype("auto", device)); tok.padding_side = "left"
directions = np.load(f"data/processed/direction_{RUN}_good_side.npz")["dirs"]

def ask_ten_times(remove_direction=False):
    text = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    enc = tok([text] * 10, return_tensors="pt", padding=True).to(device)
    hooks = [Intervention(model, [int(l)], torch.tensor(directions[int(l)]), mode="ablate") for l in LAYERS.split(",")] if remove_direction else []
    for h in hooks: h.__enter__()
    try:
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=20, do_sample=True, temperature=1.0, pad_token_id=tok.pad_token_id)
    finally:
        for h in hooks: h.__exit__(None, None, None)
    return [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

print("Question:", question)
print("Normal model        :", ask_ten_times())
print("Direction removed   :", ask_ten_times(remove_direction=True))

### 12 · Save your notes to GitHub
Copies `journal/` and `figures/` to Drive, then commits them and the hand-checked `data/scenarios.csv` to the repo (needs `GITHUB_TOKEN` in Secrets). Run it whenever you stop for the day.

In [ ]:
# Copies journal and figures to Drive (persistence), then commits them plus the hand-checked dataset to GitHub.
!mkdir -p "$DRIVE_DIR/journal" "$DRIVE_DIR/figures" && cp -r journal/. "$DRIVE_DIR/journal/" && cp -r figures/. "$DRIVE_DIR/figures/"
!git config user.email "mherje@live.com" && git config user.name "Martin Herje"
!git add journal figures data/scenarios.csv && (git commit -qm "journal, figures, scenarios: Colab session" || true) && git -c "$MATS12_GIT_AUTH" push -q origin main && echo pushed